# 00. Setup And Data Check

이 노트북은 Colab/로컬 실행 전에 한 번만 수행하는 초기 설정입니다.

- Colab이면 Google Drive를 `/drive`에 mount합니다.
- 고정 경로 `/drive/MyDrive/AI개론_박진영/lexnorm_colab_source.zip`에서 source zip을 찾습니다.
- zip을 `/content/lexnorm_submit`으로 압축 해제합니다.
- Drive의 `/drive/MyDrive/AI개론_박진영/.env`를 작업 폴더로 복제하고 환경변수로 로드합니다.
- 필요하면 `requirements.txt`를 설치합니다.
- Hugging Face dataset `weerayut/multilexnorm2026-dev-pub` 접근과 실제 language code를 확인합니다.

이후 `01`부터 `06`까지는 같은 Colab runtime을 공유한다는 전제로 실행합니다. 즉, 초기 설치/압축 해제와 환경변수 로드는 `00`에서만 합니다.


## Colab/로컬 프로젝트 루트 설정

Colab이면 Drive를 `/drive`에 mount하고 고정 zip을 `/content/lexnorm_submit`에 풉니다. 이어서 Drive의 `.env`를 복제하고 `HF_TOKEN`, `OPENAI_API_KEY`, `WANDB_API_KEY`를 현재 runtime 환경변수로 로드합니다.

주요 산출물은 각 노트북에서 local `outputs/`와 Drive `/drive/MyDrive/AI개론_박진영/lexnorm_outputs/`에 함께 저장합니다.


In [1]:
import os, shutil, zipfile
from pathlib import Path

DRIVE_ROOT = Path("/drive/MyDrive/AI개론_박진영")


def load_env_file(path):
    if not path.exists():
        return False
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip().removeprefix("export ").strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
            value = value[1:-1]
        os.environ[key] = value
    return True


def env_alias(*names):
    lowered = {key.lower(): value for key, value in os.environ.items()}
    for name in names:
        if os.environ.get(name):
            return os.environ[name]
        if lowered.get(name.lower()):
            return lowered[name.lower()]
    return None


if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive
    drive.mount("/drive", force_remount=True)
    zip_path = DRIVE_ROOT / "lexnorm_colab_source.zip"
    if not zip_path.exists():
        raise FileNotFoundError(f"missing source zip: {zip_path}")
    work_dir = Path("/content/lexnorm_submit")
    shutil.rmtree(work_dir, ignore_errors=True)
    work_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(work_dir)
    if (DRIVE_ROOT / ".env").exists():
        shutil.copy2(DRIVE_ROOT / ".env", work_dir / ".env")
    os.chdir(work_dir)

PROJECT_ROOT = Path.cwd()
env_loaded = load_env_file(PROJECT_ROOT / ".env")

hf_token = env_alias("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token", "HF_token")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

openai_key = env_alias("OPENAI_API_KEY", "openai_api_key")
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key

wandb_key = env_alias("WANDB_API_KEY", "wandb_api_key")
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key

print("PROJECT_ROOT =", PROJECT_ROOT)
print(".env loaded =", env_loaded)
print("HF_TOKEN set =", bool(os.environ.get("HF_TOKEN")))
print("OPENAI_API_KEY set =", bool(os.environ.get("OPENAI_API_KEY")))
print("WANDB_API_KEY set =", bool(os.environ.get("WANDB_API_KEY")))

if "COLAB_RELEASE_TAG" in os.environ:
    drive_output_root = DRIVE_ROOT / "lexnorm_outputs"
    drive_output_root.mkdir(parents=True, exist_ok=True)
    print("Drive output root =", drive_output_root)


Mounted at /drive
PROJECT_ROOT = /content/lexnorm_submit
.env loaded = True
HF_TOKEN set = True
OPENAI_API_KEY set = True
WANDB_API_KEY set = True
Drive output root = /drive/MyDrive/AI개론_박진영/lexnorm_outputs


## 의존성 설치

Colab runtime에서는 `00`에서만 설치합니다. 같은 runtime에서 `01` 이후 노트북을 이어서 실행하면 다시 설치할 필요가 없습니다.


In [2]:
# Colab 초기 세팅은 여기서만 수행합니다.
%pip install -q -r requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 84.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 151.3 MB/s eta 0:00:00


## 데이터셋 로드와 언어 코드 확인

Hugging Face 데이터셋을 읽고 split 구성과 실제 train language code를 출력합니다. gated dataset이므로 `HF_TOKEN`이 반드시 환경변수에 있어야 합니다.


In [3]:
from datasets import load_dataset

DATASET = "weerayut/multilexnorm2026-dev-pub"
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is not set. Put HF_TOKEN=... or hf_token=... in /drive/MyDrive/AI개론_박진영/.env and rerun 00.")

ds = load_dataset(DATASET, token=hf_token)
print(ds)
print("train languages:", sorted(set(ds["train"]["lang"])))


README.md:   0%|          | 0.00/564 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.24M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.13M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/458k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39178 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5972 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['raw', 'norm', 'lang'],
        num_rows: 39178
    })
    validation: Dataset({
        features: ['raw', 'norm', 'lang'],
        num_rows: 8408
    })
    test: Dataset({
        features: ['raw', 'norm', 'lang'],
        num_rows: 5972
    })
})
train languages: ['da', 'de', 'en', 'es', 'hr', 'id', 'iden', 'it', 'ja', 'ko', 'nl', 'sl', 'sr', 'th', 'tr', 'trde', 'vi']
